# Official fast.ai course22 original 06 — Colab adaptation

Original: [06-why-you-should-use-a-framework.ipynb](https://github.com/fastai/course22/blob/230390584ba5b990e20487d68681a9ec5b258eff/clean/06-why-you-should-use-a-framework.ipynb)

Pinned source commit: `230390584ba5b990e20487d68681a9ec5b258eff`. Source SHA256: `5e2ad07993a00617b9bbceb99b6f2a86a6b4bb3c263d054257e4d5679e921e0c`.

This additional notebook preserves the full original computational experiments. Existing independently written companions 00–07 remain separate. Set QUICK_RUN=False; run every code cell. Source notebooks and authorship belong to Jeremy Howard and fast.ai. No separate course22 repository license was found at the pinned commit; no fastbook license is applied to these files.

- Original Kaggle train/test data sourced from pinned public teaching copies. All891/418 passengers, column names, values and Name/Ticket/Cabin features verified against a second independent competition mirror; training values also equal the DataScienceDojo copy. No Kaggle provider checksum/terms acceptance/account score is claimed. Test labels remain unavailable.
- Repaired two upstream feature bugs: Title mapping now assigns its per-passenger Series instead of value_counts; Alone is true for zero relatives, since Family counts relatives. Preserved original per-table TicketFreq calculation (an identified transductive feature) and all source training schedules.

Public publication and competition submission have not been performed. Saved predictions are learning artifacts; the 418 Titanic test targets are unavailable. Original conceptual prompts/headings are retained below.


In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '23_original_06_titanic_framework.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

In [ ]:
# Full-source execution and durable evidence helpers (Colab adaptation).
import hashlib, json, shutil, urllib.request
from datetime import datetime, timezone
if QUICK_RUN or os.environ.get('COURSE_SMOKE') == '1':
    raise ValueError('These original notebooks require full mode: set QUICK_RUN=False.')
RESULTS = COLAB_CONTEXT['folder'] / 'results' / Path(COLAB_CONTEXT['run']['notebook']).stem
MODELS = COLAB_CONTEXT['folder'] / 'models' / Path(COLAB_CONTEXT['run']['notebook']).stem
WORK = COLAB_CONTEXT['folder'] / 'data' / Path(COLAB_CONTEXT['run']['notebook']).stem
for directory in [RESULTS, MODELS, WORK]: directory.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

def record_stage(stage, **evidence):
    target = RESULTS / 'stage_evidence.json'
    started = COLAB_CONTEXT['run']['started_utc']
    data = json.loads(target.read_text(encoding='utf-8')) if target.is_file() else {}
    if data.get('run_started_utc') != started:
        data = {'notebook': COLAB_CONTEXT['run']['notebook'], 'run_started_utc': started, 'stages': {}}
    data['stages'][stage] = {'recorded_utc': datetime.now(timezone.utc).isoformat(), **evidence}
    data.update(quick_run=False, status='observed_stage_outputs', provider_course_completion_claimed=False,
                learner_personal_reflection_complete=False)
    temporary = target.with_suffix('.writing')
    temporary.write_text(json.dumps(data, indent=2, default=str), encoding='utf-8')
    temporary.replace(target)

def verified_download(url, target, sha256):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.is_file():
        temporary = target.with_name(target.name + '.download')
        with urllib.request.urlopen(url, timeout=120) as response, temporary.open('wb') as out:
            shutil.copyfileobj(response, out)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != sha256:
            raise ValueError('Source download checksum mismatch: ' + url)
        temporary.replace(target)
    if hashlib.sha256(target.read_bytes()).hexdigest() != sha256:
        raise ValueError('Existing source file differs from the pinned original: ' + str(target))
    return target

def save_plot(figure, name):
    if hasattr(figure, 'write_html'):
        figure.write_html(RESULTS / (name + '.html'), include_plotlyjs=True)
    elif hasattr(figure, 'savefig'):
        figure.savefig(RESULTS / (name + '.png'), dpi=150, bbox_inches='tight')
    return figure

record_stage('full_mode_setup', execution_location='hosted_colab' if IN_COLAB else 'local_validation',
             original_exercises_verified_complete=False)

## Evidence required for the original exercises

- Execute every nonempty original code cell top to bottom in hosted Colab; retain outputs and execution records.
- QUICK_RUN=False. Original experiment data coverage, epochs and learning rates are preserved.
- Original source attribution and the per-cell provenance map distinguish original exercises from companions.
- Save full notebook outputs and actual stage/checkpoint/artifact evidence in Drive.
- No provider badge, external publishing, competition submission or personal authorship is certified by execution.
- Retain all engineered features with documented source bug repairs, original layers[10,10], LR finder slide/valley, one16-epoch lr.03 model and five16-epoch lr.03 ensemble models; export418 test predictions for standalone and ensemble.


## Introduction and set up

In [ ]:
# Exact public teaching copies of the original Kaggle competition files.
# No credentials, test labels, rule acceptance, or competition submission are inferred.
path = COLAB_CONTEXT['folder'] / 'data' / 'titanic_original'
_titanic_sources = [{'file': 'train.csv', 'url': 'https://raw.githubusercontent.com/minsuk-heo/kaggle-titanic/96c14e776090f7919d15064ed73e3b2788b4288c/input/train.csv', 'sha256': '7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f', 'rows': 891, 'columns': ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'], 'all_values_equal_independent_dsindy_pin': True, 'all_values_equal_datadojo': True}, {'file': 'test.csv', 'url': 'https://raw.githubusercontent.com/minsuk-heo/kaggle-titanic/96c14e776090f7919d15064ed73e3b2788b4288c/input/test.csv', 'sha256': '56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2', 'rows': 418, 'columns': ['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'], 'all_values_equal_independent_dsindy_pin': True}]
for _source in _titanic_sources:
    verified_download(_source['url'], path / _source['file'], _source['sha256'])
import pandas as pd
_train = pd.read_csv(path / 'train.csv')
_test = pd.read_csv(path / 'test.csv')
assert _train.shape == (891, 12) and _test.shape == (418, 11)
assert _train.PassengerId.tolist() == list(range(1, 892))
assert _test.PassengerId.tolist() == list(range(892, 1310))
assert 'Survived' not in _test and int(_train.Survived.sum()) == 342
record_stage('original_titanic_inputs', sources=_titanic_sources, train_rows=891, test_rows=418,
             held_out_test_labels_available=False, competition_submission_performed=False)

In [ ]:
from fastai.tabular.all import *

pd.options.display.float_format = '{:.2f}'.format
set_seed(42)

## Prep the data

In [ ]:
df = pd.read_csv(path/'train.csv')

In [ ]:
def add_features(df):
    df['LogFare'] = np.log1p(df['Fare'])
    df['Deck'] = df.Cabin.str[0].map(dict(A="ABC", B="ABC", C="ABC", D="DE", E="DE", F="FG", G="FG"))
    df['Family'] = df.SibSp+df.Parch
    df['Alone'] = df.Family==0  # Zero relatives means alone.
    df['TicketFreq'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['Title'] = df.Name.str.split(', ', expand=True)[1].str.split('.', expand=True)[0]
    df['Title'] = df.Title.map(dict(Mr="Mr",Miss="Miss",Mrs="Mrs",Master="Master"))

add_features(df)

assert df.Title.notna().sum() > 800, 'Title must be a per-passenger feature, not a frequency Series.'
record_stage('source_feature_engineering', columns=['LogFare','Deck','Family','Alone','TicketFreq','Title'], nonmissing_titles=int(df.Title.notna().sum()), alone_rows=int(df.Alone.sum()))

In [ ]:
splits = RandomSplitter(seed=42)(df)

In [ ]:
dls = TabularPandas(
    df, splits=splits,
    procs = [Categorify, FillMissing, Normalize],
    cat_names=["Sex","Pclass","Embarked","Deck", "Title"],
    cont_names=['Age', 'SibSp', 'Parch', 'LogFare', 'Alone', 'TicketFreq', 'Family'],
    y_names="Survived", y_block = CategoryBlock(),
).dataloaders(path=".")

## Train the model

In [ ]:
learn = tabular_learner(dls, metrics=accuracy, layers=[10,10])

In [ ]:
learn.lr_find(suggest_funcs=(slide, valley))

In [ ]:
learn.fit(16, lr=0.03)

learn.export(MODELS / 'standalone.pkl')
record_stage('standalone_framework_training', requested_epochs=16, learning_rate=.03, layers=[10,10], validation_metrics=[float(v) for v in learn.validate()], exported_model=str(MODELS / 'standalone.pkl'))

## Submit to Kaggle

In [ ]:
tst_df = pd.read_csv(path/'test.csv')
tst_df['Fare'] = tst_df.Fare.fillna(0)
add_features(tst_df)

In [ ]:
tst_dl = learn.dls.test_dl(tst_df)

In [ ]:
preds,_ = learn.get_preds(dl=tst_dl)

In [ ]:
tst_df['Survived'] = (preds[:,1]>0.5).int()
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv(RESULTS / 'sub.csv', index=False)
record_stage('standalone_test_predictions', rows=len(sub_df), csv=str(RESULTS / 'sub.csv'), submitted=False)

In [ ]:
pd.read_csv(RESULTS / 'sub.csv').head()

## Ensembling

In [ ]:
_ensemble_index = 0
def ensemble():
    global _ensemble_index
    _ensemble_index += 1
    learn = tabular_learner(dls, metrics=accuracy, layers=[10,10])
    with learn.no_bar(),learn.no_logging(): learn.fit(16, lr=0.03)
    learn.export(MODELS / f'ensemble_{_ensemble_index}.pkl')
    result = learn.get_preds(dl=tst_dl)[0]
    assert result.shape == (418, 2)
    record_stage(f'ensemble_model_{_ensemble_index}', requested_epochs=16, learning_rate=.03,
                 layers=[10,10], validation_metrics=[float(v) for v in learn.validate()], test_prediction_rows=len(result))
    return result

In [ ]:
learns = [ensemble() for _ in range(5)]

In [ ]:
ens_preds = torch.stack(learns).mean(0)

In [ ]:
tst_df['Survived'] = (ens_preds[:,1]>0.5).int()
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv(RESULTS / 'ens_sub.csv', index=False)
record_stage('five_model_test_ensemble', models=len(learns), rows=len(sub_df), csv=str(RESULTS / 'ens_sub.csv'), submitted=False)

## Final thoughts

In [ ]:
record_stage('reached_final_cell', all_previous_cells_independently_verified=False)
finish_run(COLAB_CONTEXT)